**Cell 1**

# 05 — Evaluation

Evaluate the trained YOLOv11 model on the held-out **test** split:
- **Precision**, **Recall**
- **mAP@0.5**, **mAP@0.5:0.95**
- **Confusion matrix** with per-class breakdown
- **PR curves** per class
- **Comparison table** across YOLOv11 variants
- Qualitative prediction grid

In [ ]:
# Cell 2 — install dependencies for this notebook
%pip install -q "ultralytics==8.3.*" pandas numpy matplotlib seaborn pillow
# On a bare Linux box (no system libGL) OpenCV import can fail — if so, uncomment:
# !apt-get update -qq && apt-get install -y -qq libgl1

In [ ]:
# Cell 3 — imports
from ultralytics import YOLO
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
DATA_YAML = Path('../data/dataset/data.yaml').resolve()
WEIGHTS = Path('../weights/best.pt').resolve()
assert WEIGHTS.exists(), 'train the model in notebook 04 first'

**Cell 4**

## 5.1 Run validation on the test split

In [ ]:
# Cell 5 — ultralytics .val() on the test split
model = YOLO(WEIGHTS)
metrics = model.val(data=str(DATA_YAML), split='test', imgsz=640, conf=0.25, iou=0.5)
print('mAP@0.5      :', round(metrics.box.map50, 4))
print('mAP@0.5:0.95 :', round(metrics.box.map,   4))
print('Precision (macro):', round(metrics.box.mp, 4))
print('Recall    (macro):', round(metrics.box.mr, 4))

**Cell 6**

## 5.2 Per-class metrics table

In [ ]:
# Cell 7 — per-class precision / recall / mAP
p, r, ap50, ap = metrics.box.p, metrics.box.r, metrics.box.ap50, metrics.box.ap
per_class = pd.DataFrame({
    'class': CLASSES,
    'precision': np.round(p, 4),
    'recall':    np.round(r, 4),
    'mAP@0.5':   np.round(ap50, 4),
    'mAP@0.5:0.95': np.round(ap.mean(1) if ap.ndim > 1 else ap, 4),
})
per_class

**Cell 8**

## 5.3 Confusion matrix (absolute + normalized)

In [ ]:
# Cell 9 — plot the confusion matrix Ultralytics computes
cm = metrics.confusion_matrix.matrix  # shape: (C+1, C+1), last row/col = background
labels = CLASSES + ['background']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='.0f', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax[0])
ax[0].set_title('Confusion matrix (counts)'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')

cm_norm = cm / cm.sum(axis=0, keepdims=True).clip(min=1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax[1])
ax[1].set_title('Confusion matrix (column-normalized)'); ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
plt.tight_layout(); plt.show()

**Cell 10**

## 5.4 Per-class PR curves

In [ ]:
# Cell 11 — Ultralytics writes PR_curve.png under the val run
pr_img = Path(metrics.save_dir) / 'PR_curve.png'
if pr_img.exists():
    plt.figure(figsize=(8, 6)); plt.imshow(Image.open(pr_img)); plt.axis('off'); plt.show()
else:
    print('PR_curve.png not found at', pr_img)

**Cell 12**

## 5.5 Model-variant comparison table

Re-run cells 5–7 after training each variant (see notebook 04, cell 5: change `model='yolo11n.pt' | 'yolo11s.pt' | 'yolo11m.pt'`) and fill this table.

In [ ]:
# Cell 13 — placeholder comparison table; overwrite with real numbers from each run
comparison = pd.DataFrame([
    {'variant': 'yolo11n', 'params_M': 2.6,  'mAP@0.5': None, 'mAP@0.5:0.95': None, 'latency_ms': None},
    {'variant': 'yolo11s', 'params_M': 9.4,  'mAP@0.5': round(metrics.box.map50, 4), 'mAP@0.5:0.95': round(metrics.box.map, 4), 'latency_ms': None},
    {'variant': 'yolo11m', 'params_M': 20.0, 'mAP@0.5': None, 'mAP@0.5:0.95': None, 'latency_ms': None},
])
comparison

**Cell 14**

## 5.6 Qualitative prediction grid

In [ ]:
# Cell 15 — show predictions on 8 random test images
import random
test_imgs = sorted(Path('../data/dataset/images/test').glob('*.jpg'))
sample = random.sample(test_imgs, min(8, len(test_imgs)))
preds = model.predict(source=[str(p) for p in sample], imgsz=640, conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.flatten(), preds):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.set_title(Path(r.path).name, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

**Cell 16**

### Reporting checklist
- [ ] Per-class precision / recall / mAP table filled with real numbers
- [ ] Confusion matrix inspected — identify dominant confusion pairs
- [ ] PR curves attached
- [ ] Variant comparison table completed for `yolo11n`, `yolo11s`, `yolo11m`
- [ ] 8-image qualitative grid included in the report